# Multi-Service Credential Management

**Scenario:** An analytics pipeline needs credentials for Google Analytics,
a PostgreSQL database, and an S3 bucket. Each credential lives in a different
backend: 1Password for service accounts, environment variables for CI,
Apple Keychain for local dev.

siege_utilities provides a unified `CredentialManager` that searches all
backends in priority order and reports what's available.

## What this shows

Discover configured credential backends and demonstrate missing-credential/error-handling patterns without requiring real secrets. This is a pure/offline notebook: it should run without network credentials, GDAL, Spark, or external services.


## 1. Offline Credential Backend Discovery

Before configuring anything, check which backends are available on this
machine. The manager detects 1Password CLI, Apple Keychain, environment
variables, and filesystem credential files.

In [ ]:
from siege_utilities.config.credential_manager import CredentialManager

# Offline-safe manager: this notebook demonstrates credential call shapes
# without touching 1Password, Keychain, prompts, network services, or real secrets.
manager = CredentialManager(
    backend_priority=["files", "env"],
    credential_paths=[],
    detect_backends=False,
    include_default_paths=False,
    create_default_paths=False,
)

print("Offline credential backend priority:")
for name in manager.backend_priority:
    print(f"- {name}: {'available' if manager.available_backends.get(name) else 'unavailable'}")


## 2. The CredentialManager Class

The manager can be limited to offline-safe backends for demos and CI. It provides a
unified `get_credential()` method. It searches backends in priority order:
files -> environment in this notebook, while production profiles may opt into 1Password, Keychain, or prompts outside CI.

When a credential isn't found, it raises `CredentialNotFoundError` —
never returns None or empty string (SU-1: errors are not data).

In [ ]:
from siege_utilities.config.credential_manager import (
    CredentialManager, CredentialNotFoundError
)

manager = CredentialManager(
    backend_priority=["files", "env"],
    credential_paths=[],
    detect_backends=False,
    include_default_paths=False,
    create_default_paths=False,
)

print(f"Manager initialized")
print(f"Backend priority: {manager.backend_priority}")

# Missing credentials raise, not return empty
try:
    manager.get_credential("nonexistent-service", "fake-user")
except CredentialNotFoundError as e:
    print(f"\nMissing credential raises: {type(e).__name__}")
    print(f"  (SU-1: errors are not data)")

## 3. Environment Variable Pattern

In CI/CD, credentials come from environment variables. The manager
checks for them automatically. This is the pattern used in GitHub Actions
and Databricks secret scopes.

In [ ]:
import os

# Demonstrate the env-var credential pattern
# In production, these would be set by CI or the shell profile
os.environ["SIEGE_TEST_DB_HOST"] = "localhost"
os.environ["SIEGE_TEST_DB_PORT"] = "5432"

# The pattern: check env first, fall back to credential manager
db_host = os.environ.get("SIEGE_TEST_DB_HOST", "not-set")
db_port = os.environ.get("SIEGE_TEST_DB_PORT", "not-set")

print(f"DB Host: {db_host}")
print(f"DB Port: {db_port}")
print(f"\nPattern: os.environ.get() for CI, CredentialManager.get_credential() for local dev")

# Clean up
del os.environ["SIEGE_TEST_DB_HOST"]
del os.environ["SIEGE_TEST_DB_PORT"]

## Key Patterns

- **Offline-safe demos** — construct `CredentialManager(
    backend_priority=["files", "env"],
    credential_paths=[],
    detect_backends=False,
    include_default_paths=False,
    create_default_paths=False,
)`
- **Production opt-in** — add 1Password, Keychain, or prompt backends only outside CI/non-interactive notebooks
- **CredentialNotFoundError** — missing credentials raise, never return empty (SU-1)
- **Default manager priority**: files -> env -> 1Password -> Keychain -> prompt. Notebooks use `backend_priority=["files", "env"]` for offline-safe execution; production callers may override the order explicitly.
- **siege_zsh integration** — when available, shell profile pre-loads credentials
  into the environment so the manager finds them without interactive prompts

## Related

- Source: `siege_utilities/config/credential_manager.py`
- Tests: `tests/test_credential_manager.py`, `tests/test_config_errors.py`
- Notebook governance: `tests/test_notebook_hygiene.py`, `tests/test_notebooks.py`, `scripts/check_notebook_inventory.py`
